### The Smart Supplier: Optimizing Orders in a Fluctuating Market

Develop a reinforcement learning agent using dynamic programming to help a Smart Supplier decide which products to manufacture and sell each day to maximize profit. The agent must learn the optimal policy for choosing daily production quantities, considering its limited raw materials and the unpredictable daily demand and selling prices for different products.

#### **Scenario**
 A small Smart Supplier manufactures two simple products: Product A and Product B. Each day, the supplier has a limited amount of raw material. The challenge is that the market demand and selling price for Product A and Product B change randomly each day, making some products more profitable than others at different times. The supplier needs to decide how much of each product to produce to maximize profit while managing their limited raw material.

#### **Objective**
The Smart Supplier's agent must learn the optimal policy π∗ using dynamic programming (Value Iteration or Policy Iteration) to decide how many units of Product A and Product B to produce each day to maximize the total profit over the fixed number of days, given the daily changing market conditions and limited raw material.

## GROUP 107 TEAM MEMBERS
| Name | BITS ID|
|------| -------|
|SUKAMATI NAVEEN KUMAR  | **2024AA05078** |
|MANISH GARG |**2024AA05118** |
|GAURAV DHAMI | **2024AA05077** |
|HARSHIT KUMAR HALWAN | **2024AA05928** |

## 1. Custom Environment Creation (SmartSupplierEnv)

* The environment is fully known: we can simulate rewards and transitions.

* The action space is finite and discrete.

* The horizon (5 days) is limited, allowing backward planning.

* We needed to find a deterministic, optimal policy for each state.

In [15]:
class SmartSupplierEnv:
    def __init__(self):
        """
        Initializes the Smart Supplier environment.
        - Sets daily raw material limit and number of days for planning.
        - Defines possible market states with prices for Product A and B.
        - Specifies all possible production actions and their raw material costs.
        """
        self.initial_raw_material = 10  # Maximum raw material available per day
        self.num_days = 5  # Number of days to optimize production
        self.market_states = {
            1: {'A': 8, 'B': 2},  # Market State 1: High demand for A
            2: {'A': 3, 'B': 5}   # Market State 2: High demand for B
        }
        # Actions: (units of A, units of B, total raw material cost)
        self.actions = [
            (2, 0, 4),  # Produce 2A, 0B
            (1, 2, 4),  # Produce 1A, 2B
            (0, 5, 5),  # Produce 0A, 5B
            (3, 0, 6),  # Produce 3A, 0B
            (0, 0, 0)   # Do nothing
        ]
        self.action_names = [
            "Produce_2A_0B", "Produce_1A_2B", "Produce_0A_5B", "Produce_3A_0B", "Do_Nothing"
        ]

    def get_reward(self, action_id, market_state):
        """
        Calculates the immediate reward for taking an action in a given market state.
        - Multiplies produced units by their respective market prices.
        - Returns total profit for the action.
        """
        a, b, _ = self.actions[action_id]
        prices = self.market_states[market_state]
        return a * prices['A'] + b * prices['B']

    def is_action_valid(self, action_id, rm_left):
        """
        Checks if the selected action is feasible given remaining raw material.
        - Compares action's raw material cost to available supply.
        - Returns True if action can be performed, else False.
        """
        _, _, cost = self.actions[action_id]
        return cost <= rm_left

    def get_possible_states(self):
        """
        Generator for all possible environment states.
        - Iterates through each day, possible raw material levels, and market states.
        - Yields (day, raw material, market state) tuples for use in DP algorithms.
        """
        for day in range(1, self.num_days + 1):
            for rm in range(self.initial_raw_material + 1):
                for market in [1, 2]:
                    yield (day, rm, market)



- We built a custom environment to simulate the Smart Supplier scenario with real-world constraints.
- In this class, we defined raw material usage, production actions, reward logic, and the two possible market states.
- This environment provides valid states, calculates rewards, and checks whether actions are feasible.


## 2. Dynamic Programming Implementation (Value Iteration or Policy Iteration)

In [16]:
def value_iteration(env, gamma=1.0, theta=1e-5):
    """
    Performs Value Iteration to compute the optimal policy and state-value function.
    - Iterates backward from the last day to the first, updating value estimates.
    - For each state, selects the action with the highest expected return.
    - Considers both immediate rewards and expected future profits.
    Returns:
        V: Dictionary mapping states to their optimal value.
        policy: Dictionary mapping states to the best action.
    """
    V = defaultdict(float)  # Stores value of each state
    policy = dict()         # Stores best action for each state

    for day in reversed(range(1, env.num_days + 1)):
        for rm in range(env.initial_raw_material + 1):
            for market in [1, 2]:
                state = (day, rm, market)
                max_value = -float('inf')
                best_action = None
                for action_id in range(len(env.actions)):
                    if not env.is_action_valid(action_id, rm):
                        continue
                    reward = env.get_reward(action_id, market)
                    if day == env.num_days:
                        total_value = reward  # No future rewards on final day
                    else:
                        next_day = day + 1
                        next_rm = env.initial_raw_material
                        # Market state is random: average over both possibilities
                        next_v = 0.5 * V[(next_day, next_rm, 1)] + 0.5 * V[(next_day, next_rm, 2)]
                        total_value = reward + gamma * next_v
                    if total_value > max_value:
                        max_value = total_value
                        best_action = action_id
                if best_action is None:
                    best_action = len(env.actions) - 1  # Default to Do_Nothing
                    max_value = 0
                V[state] = max_value
                policy[state] = best_action
    return V, policy



- We implemented **Value Iteration** to calculate the optimal state-value function `V*` and policy `π*`.
- Our value updates consider both immediate profit and expected future rewards across random market changes.
- We iterated backwards from Day 5 to Day 1 to ensure future-aware decisions.
- This gave us the optimal action to take at every state in our environment.

## 3. Simulation and Policy Analysis

In [17]:
# simulate policy function - Simulates the learned policy over multiple runs to evaluate performance
def simulate_policy(env, policy, num_runs=1000):
    """
    Simulates the learned policy over multiple episodes to estimate performance.
    - For each run, simulates 5 days of production with random market states.
    - Tracks total profit for each episode.
    - Returns average and standard deviation of profits across all runs.
    """
    total_profits = []
    for _ in range(num_runs):
        profit = 0
        day = 1
        rm = env.initial_raw_material
        market = random.choice([1, 2])
        for d in range(1, env.num_days + 1):
            state = (day, rm, market)
            action_id = policy.get(state, len(env.actions) - 1)
            if not env.is_action_valid(action_id, rm):
                action_id = len(env.actions) - 1  # Do_Nothing if invalid
            reward = env.get_reward(action_id, market)
            profit += reward
            day += 1
            rm = env.initial_raw_material
            market = random.choice([1, 2])
        total_profits.append(profit)
    avg_profit = np.mean(total_profits)
    std_profit = np.std(total_profits)
    return avg_profit, std_profit




- After learning the optimal policy, we simulated the policy over 1000 episodes of 5 days each.
- We recorded the total profit for each simulation and computed the average and standard deviation.
- This helped us verify that the policy is not just theoretically optimal, but also performs well

# 4. Analyze Policy for Key States

In [18]:
def analyze_policy(env, policy):
    """
    Prints the optimal action for every possible state.
    - Loops through all days, raw material levels, and market states.
    - Displays which action the policy recommends for each situation.
    - Helps visualize and verify the learned policy's logic.
    """
    print("### Full Optimal Policy Table (Day, RM, Market State) ###\n")
    for day in range(1, 6):  # Days 1 to 5
        for rm in range(0, 11):  # RM 0 to 10
            for market in [1, 2]:  # Market State 1 and 2
                action = policy.get((day, rm, market), None)
                action_name = env.action_names[action] if action is not None else "Invalid"
                print(f"Day {day}, RM {rm}, Market {market} → {action_name}")


- We analyzed the learned policy by observing its decisions in key states (like Day 1 with 10 RM in both markets).
- We found that the policy adapts based on:
  - Which product is more profitable in the current market,
  - How much raw material is left,
  - How close we are to the final day.
- This confirmed that our agent is behaving intelligently under different scenarios

## 5. Impact of Dynamics Analysis

In [19]:
def compare_fixed_vs_dynamic(env, policy_dynamic):
    """
    Compares the dynamic policy to a baseline fixed-market policy.
    - Computes a fixed policy assuming market is always in State 1.
    - Simulates both policies over 1000 runs and compares average profits.
    - Demonstrates the advantage of adapting to dynamic market conditions.
    """
    fixed_policy = dict()
    V_fixed = defaultdict(float)
    for day in reversed(range(1, env.num_days + 1)):
        for rm in range(env.initial_raw_material + 1):
            state = (day, rm)
            max_value = -float('inf')
            best_action = None
            for action_id in range(len(env.actions)):
                if not env.is_action_valid(action_id, rm):
                    continue
                reward = env.get_reward(action_id, 1)
                if day == env.num_days:
                    total_value = reward
                else:
                    next_day = day + 1
                    next_rm = env.initial_raw_material
                    total_value = reward + V_fixed[(next_day, next_rm)]
                if total_value > max_value:
                    max_value = total_value
                    best_action = action_id
            if best_action is None:
                best_action = len(env.actions) - 1
                max_value = 0
            V_fixed[state] = max_value
            fixed_policy[state] = best_action

    profits = []
    for _ in range(1000):
        profit = 0
        day = 1
        rm = env.initial_raw_material
        for d in range(1, env.num_days + 1):
            state = (day, rm)
            action_id = fixed_policy.get(state, len(env.actions) - 1)
            if not env.is_action_valid(action_id, rm):
                action_id = len(env.actions) - 1
            reward = env.get_reward(action_id, 1)
            profit += reward
            day += 1
            rm = env.initial_raw_material
        profits.append(profit)

    avg_fixed = np.mean(profits)
    avg_dynamic, _ = simulate_policy(env, policy_dynamic, 1000)
    print(f"Avg profit (fixed Market State 1): {avg_fixed:.2f}")
    print(f"Avg profit (dynamic/fluctuating): {avg_dynamic:.2f}")



- We compared our learned policy with a baseline policy assuming a fixed market (always Market State 1).
- Our dynamic policy outperformed the fixed one by adapting to daily market shifts.
- This demonstrated the strength of reinforcement learning in environments with uncertainty and non-stationarity.


## 6. Main execution

In [20]:
import random
import numpy as np
from collections import defaultdict

# --- Main Execution ---
if __name__ == "__main__":
    # Sets up environment and runs value iteration to learn the optimal policy
    env = SmartSupplierEnv()
    V, policy = value_iteration(env)
    print("Optimal policy learned using Value Iteration.\n")

    # Outputs the full policy table for all states
    analyze_policy(env, policy)

    # Simulates the learned policy and reports average profit
    avg_profit, std_profit = simulate_policy(env, policy, 1000)
    print(f"\nAverage profit over 1000 simulations: {avg_profit:.2f} ± {std_profit:.2f}")

    # Compares dynamic and fixed-market policies
    print("\n--- Impact of Market Dynamics ---")
    compare_fixed_vs_dynamic(env, policy)

    # Prints value function for key states as additional analysis
    print('\n--- V* (State Values) for Key States ---')
    for key_state in [(1, 10, 1), (1, 10, 2), (5, 10, 1), (5, 10, 2), (3, 4, 1), (3, 4, 2)]:
        print(f"V*{key_state} = {V[key_state]:.2f}")

Optimal policy learned using Value Iteration.

### Full Optimal Policy Table (Day, RM, Market State) ###

Day 1, RM 0, Market 1 → Do_Nothing
Day 1, RM 0, Market 2 → Do_Nothing
Day 1, RM 1, Market 1 → Do_Nothing
Day 1, RM 1, Market 2 → Do_Nothing
Day 1, RM 2, Market 1 → Do_Nothing
Day 1, RM 2, Market 2 → Do_Nothing
Day 1, RM 3, Market 1 → Do_Nothing
Day 1, RM 3, Market 2 → Do_Nothing
Day 1, RM 4, Market 1 → Produce_2A_0B
Day 1, RM 4, Market 2 → Produce_1A_2B
Day 1, RM 5, Market 1 → Produce_2A_0B
Day 1, RM 5, Market 2 → Produce_0A_5B
Day 1, RM 6, Market 1 → Produce_3A_0B
Day 1, RM 6, Market 2 → Produce_0A_5B
Day 1, RM 7, Market 1 → Produce_3A_0B
Day 1, RM 7, Market 2 → Produce_0A_5B
Day 1, RM 8, Market 1 → Produce_3A_0B
Day 1, RM 8, Market 2 → Produce_0A_5B
Day 1, RM 9, Market 1 → Produce_3A_0B
Day 1, RM 9, Market 2 → Produce_0A_5B
Day 1, RM 10, Market 1 → Produce_3A_0B
Day 1, RM 10, Market 2 → Produce_0A_5B
Day 2, RM 0, Market 1 → Do_Nothing
Day 2, RM 0, Market 2 → Do_Nothing
Day 2, RM 

## Summary of Smart Supplier

### Final Summary: Team Submission (Group 107)

---

#### 1. **Custom Environment Creation**
- Designed `SmartSupplierEnv` with raw material limits, product costs, and random daily market shifts.
- Implemented state space, reward logic, and five production actions including a "Do Nothing" option.

---

#### 2. **Dynamic Programming Implementation**
- Applied Value Iteration to compute the optimal policy over 5 days using expected future rewards.
- States defined by (Day, RM, Market) and policy maps each state to the best valid production action.

---

#### 3. **Optimal Policy Analysis**
- Policy adapts to market state: favors Product A in Market 1, Product B in Market 2.
- Uses cheaper actions when RM is low, and becomes aggressive on Day 5 to maximize immediate reward.

---

#### 4. **Performance Evaluation**
- Extracted and reported V* values for key states like (1,10,1), (5,10,2), and (3,4,1).
- Ran 1000 simulations and computed consistent average profit, confirming policy effectiveness.

---

#### 5. **Impact of Dynamics**
- Compared learned policy to a fixed-price baseline (always Market 1); dynamic policy performed better.
- RL agent effectively adapted to uncertainty, showing benefits over static rule-based strategies.

---

### Impact of Dynamics

In our assignment, we compared the optimal policy learned in a **dynamic environment** (where the market changes randomly each day) to a policy learned under a **fixed market condition** (where the market is always in State 1).

#How does the agent's strategy adapt or change when the market can shift unexpectedly, versus if it were always the same?

#### Fixed vs Dynamic Market Comparison

- In the **fixed market (always State 1)** setup:
  - The agent consistently chose actions that maximize profit based on Product A's higher value (8) and lower reward for Product B (2).
  - Since the environment is predictable, the policy becomes repetitive and focuses on a narrow action set favoring Product A.

- In the **dynamic market** setup:
  - Our agent had to plan for both possible market states (State 1 or State 2), each equally likely.
  - The policy learned to balance risk and reward: producing Product A when high reward was expected and switching to Product B when more profitable.
  - It dynamically adjusted its strategy depending on the day, available raw material, and market condition, demonstrating a clear advantage in adaptability.

#### Result of Adaptation

- From simulations, we observed that the **dynamic policy outperformed** the fixed policy in terms of average profit over 1000 runs.
- The agent learned to **exploit market opportunities** and **minimize losses** under uncertainty, rather than relying on static assumptions.

#### final observation

By comparing both scenarios, we saw that **incorporating dynamic market behavior into policy learning results in smarter, more flexible decision-making**. Our agent was able to generalize better and extract more value from the environment compared to a rigid, one-size-fits-all policy.



